# Obsolete Filter — Loại bỏ Văn bản Hết hiệu lực

Notebook này thêm **bước 1.6** vào pipeline, chạy **sau** `preprocessData.ipynb` và **trước** `filterData.ipynb`.

## Mục tiêu
Loại bỏ các điều khoản thuộc văn bản **đã hết hiệu lực / bị thay thế** khỏi `legal_dataset_processed.parquet`
để tránh RAG truy xuất ra pháp luật cũ, sai.

## Hai lớp phát hiện
1. **Auto-detect từ content**: quét nội dung tìm các cụm từ pháp lý chuẩn như
   *"thay thế ... Nghị định số 110/2004"*, *"bãi bỏ Thông tư số ..."*, v.v.
2. **Hardcode blacklist**: danh sách tĩnh các văn bản quan trọng đã biết chắc là hết hiệu lực
   (ưu tiên cho lĩnh vực hành chính chung).

## Vị trí trong pipeline
```
preprocessData.ipynb  → legal_dataset_processed.parquet (392k hàng)
       ↓
obsoleteFilter.ipynb  → legal_dataset_active.parquet    (~?k hàng)  ← notebook này
       ↓
filterData.ipynb      → legal_dataset_filtered.parquet  (~?k hàng)
       ↓
chunkData.ipynb       → legal_chunks.parquet
```

> **Ghi chú về độ chính xác:** Auto-detect dựa trên NLP pattern — có thể có false positive/negative.
> Luôn review `obsolete_review.csv` sau khi chạy trước khi tiếp tục pipeline.

---
## 0. Setup

In [2]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

#DATASET_DIR  = Path("../dataset/processed")
DATASET_DIR  = Path("dataset/processed")
INPUT_PATH   = DATASET_DIR / "legal_dataset_processed.parquet"
OUTPUT_PATH  = DATASET_DIR / "legal_dataset_active.parquet"
REVIEW_PATH  = DATASET_DIR / "obsolete_review.csv"    # để human review
REPORT_PATH  = DATASET_DIR / "obsolete_report.txt"    # báo cáo tổng hợp

df = pd.read_parquet(INPUT_PATH, engine="pyarrow")
print(f"Input: {len(df):,} hàng")
print(f"Cột  : {list(df.columns)}")

Input: 392,079 hàng
Cột  : ['doc_id', 'id', 'ministry', 'type_normalized', 'name', 'chapter_id', 'chapter_name', 'article', 'content']


---
## 1. Hardcode Blacklist

Danh sách văn bản **chắc chắn hết hiệu lực** — match theo cột `id` (số hiệu văn bản).

Ưu tiên lĩnh vực **hành chính chung, văn thư, tổ chức bộ máy, cán bộ công chức**.
Mỗi entry gồm: số hiệu, lý do ngắn, văn bản thay thế.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# BLACKLIST — văn bản đã hết hiệu lực
# Format: ("số_hiệu_regex", "lý_do", "văn_bản_thay_thế")
# Dùng regex để match linh hoạt (VD: "110/2004" sẽ khớp "110/2004/NĐ-CP")
# ─────────────────────────────────────────────────────────────────────────────
BLACKLIST = [

    # ── CÔNG TÁC VĂN THƯ / LƯU TRỮ ─────────────────────────────────────────
    (r"110/2004/NĐ-CP",       "Thay bởi NĐ 30/2020",            "30/2020/NĐ-CP"),
    (r"09/2010/NĐ-CP",        "Thay bởi NĐ 30/2020",            "30/2020/NĐ-CP"),
    (r"04/2013/TT-BNV",       "Thay bởi TT 01/2019/TT-BNV",     "01/2019/TT-BNV"),
    (r"07/2012/TT-BNV",       "Thay bởi TT 01/2019/TT-BNV",     "01/2019/TT-BNV"),
    (r"01/2011/TT-BNV",       "Thay bởi NĐ 30/2020 + TT mới",   "30/2020/NĐ-CP"),
    (r"55/2005/NĐ-CP",        "Thay bởi NĐ 110/2004 → 30/2020", "30/2020/NĐ-CP"),
    (r"02/2012/TT-BNV",       "TT lưu trữ cũ — thay bởi TT 02/2021/TT-BNV", "02/2021/TT-BNV"),
    (r"09/2007/NĐ-CP",        "NĐ lưu trữ 2007 — thay bởi NĐ 30/2020",      "30/2020/NĐ-CP"),

    # ── TỔ CHỨC BỘ MÁY / CHÍNH PHỦ ─────────────────────────────────────────
    (r"36/2012/NĐ-CP",        "Thay bởi NĐ 138/2016/NĐ-CP",     "138/2016/NĐ-CP"),
    (r"178/2007/NĐ-CP",       "Thay bởi NĐ 138/2016/NĐ-CP",     "138/2016/NĐ-CP"),
    (r"132/2007/NĐ-CP",       "Thay bởi NĐ 45/2016/NĐ-CP",      "45/2016/NĐ-CP"),
    (r"15/2008/NĐ-CP",        "NĐ bộ ngành 2008 — thay bởi NĐ 123/2016",    "123/2016/NĐ-CP"),
    (r"08/2012/NĐ-CP",        "Thay bởi NĐ 123/2016/NĐ-CP",     "123/2016/NĐ-CP"),

    # ── CÁN BỘ, CÔNG CHỨC, VIÊN CHỨC ────────────────────────────────────────
    (r"117/2003/NĐ-CP",       "Thay bởi NĐ 29/2012/NĐ-CP",      "29/2012/NĐ-CP"),
    (r"116/2003/NĐ-CP",       "Thay bởi NĐ 18/2010/NĐ-CP",      "18/2010/NĐ-CP"),
    (r"115/2003/NĐ-CP",       "Thay bởi NĐ 06/2010/NĐ-CP",      "06/2010/NĐ-CP"),
    (r"54/2005/NĐ-CP",        "Thay bởi NĐ 34/2011/NĐ-CP",      "34/2011/NĐ-CP"),
    (r"35/2005/NĐ-CP",        "Thay bởi NĐ 34/2011/NĐ-CP",      "34/2011/NĐ-CP"),
    (r"06/2006/NĐ-CP",        "Thay bởi NĐ 107/2020/NĐ-CP",     "107/2020/NĐ-CP"),
    (r"24/2010/NĐ-CP",        "NĐ tuyển dụng CBCC cũ — thay bởi NĐ 138/2020", "138/2020/NĐ-CP"),
    (r"18/2010/NĐ-CP",        "NĐ viên chức cũ — thay bởi NĐ 115/2020",      "115/2020/NĐ-CP"),
    (r"29/2012/NĐ-CP",        "Thay bởi NĐ 138/2020/NĐ-CP",     "138/2020/NĐ-CP"),
    (r"41/2012/NĐ-CP",        "NĐ vị trí việc làm cũ — thay bởi NĐ 62/2020", "62/2020/NĐ-CP"),
    (r"36/2013/NĐ-CP",        "NĐ vị trí việc làm VC cũ — thay bởi NĐ 106/2020", "106/2020/NĐ-CP"),
    (r"112/2011/NĐ-CP",       "NĐ CBCC xã cũ — thay bởi NĐ 34/2019/NĐ-CP", "34/2019/NĐ-CP"),

    # ── BAN HÀNH VĂN BẢN QPPL ───────────────────────────────────────────────
    (r"17/1997/QH10",         "Luật BHVBQPPL 1997 — thay bởi 80/2015",       "80/2015/QH13"),
    (r"02/2002/QH11",         "Luật BHVBQPPL 2002 — thay bởi 80/2015",       "80/2015/QH13"),
    (r"17/2008/QH12",         "Luật BHVBQPPL 2008 — thay bởi 80/2015",       "80/2015/QH13"),
    (r"161/2005/NĐ-CP",       "NĐ hướng dẫn BH VBQPPL cũ — thay bởi NĐ 34/2016", "34/2016/NĐ-CP"),
    (r"24/2009/NĐ-CP",        "NĐ hướng dẫn BHVBQPPL cũ — thay bởi NĐ 34/2016",  "34/2016/NĐ-CP"),

    # ── TỔ CHỨC CHÍNH QUYỀN ĐỊA PHƯƠNG ─────────────────────────────────────
    (r"11/2003/QH11",         "Luật TCCQĐP cũ — thay bởi 77/2015/QH13",      "77/2015/QH13"),

    # ── LAO ĐỘNG, TIỀN LƯƠNG ────────────────────────────────────────────────
    (r"10/2012/QH13",         "Bộ luật LĐ 2012 — thay bởi 45/2019/QH14",     "45/2019/QH14"),
    (r"35/1994/QH9",          "Bộ luật LĐ 1994 — thay bởi 45/2019/QH14",     "45/2019/QH14"),
    (r"74/2006/QH11",         "Luật BHXH 2006 — thay bởi 58/2014/QH13",      "58/2014/QH13"),
    (r"152/2006/NĐ-CP",       "NĐ BHXH 2006 — thay bởi NĐ 115/2015/NĐ-CP",  "115/2015/NĐ-CP"),
    (r"68/2007/NĐ-CP",        "NĐ tiền lương CBCC cũ — thay bởi NĐ 38/2019", "38/2019/NĐ-CP"),

    # ── KHIẾU NẠI, TỐ CÁO ───────────────────────────────────────────────────
    (r"09/1998/QH10",         "Luật KN,TC 1998 — thay bởi 02/2011/QH13",     "02/2011/QH13"),
    (r"26/2004/QH11",         "Sửa Luật KN,TC 2004 — thay bởi 02/2011",      "02/2011/QH13"),

    # ── PHÒNG CHỐNG THAM NHŨNG ──────────────────────────────────────────────
    (r"55/2005/QH11",         "Luật PCTN 2005 — thay bởi 36/2018/QH14",      "36/2018/QH14"),
    (r"01/2007/QH12",         "Sửa Luật PCTN 2007 — thay bởi 36/2018",       "36/2018/QH14"),
    (r"27/2012/QH13",         "Sửa Luật PCTN 2012 — thay bởi 36/2018",       "36/2018/QH14"),

    # ── HÀNH CHÍNH CÔNG / ĐẦU TƯ ────────────────────────────────────────────
    (r"22/2004/NĐ-CP",        "NĐ thanh tra cũ — thay bởi NĐ 86/2011/NĐ-CP", "86/2011/NĐ-CP"),
    (r"65/2017/NĐ-CP",        "NĐ cơ chế một cửa cũ — thay bởi NĐ 61/2018",  "61/2018/NĐ-CP"),
    (r"57/2012/NĐ-CP",        "NĐ chế độ thông tin BC cũ — thay bởi NĐ 09/2019", "09/2019/NĐ-CP"),

    # ── THÊM VÀO ĐÂY KHI CẦN ────────────────────────────────────────────────
    # (r"<số_hiệu_regex>",   "<lý_do>",  "<văn_bản_thay_thế>"),
]

print(f"Blacklist: {len(BLACKLIST)} văn bản")


Blacklist: 44 văn bản


---
## 2. Auto-detect từ Content

Quét toàn bộ `content` tìm các cụm từ pháp lý báo hiệu **văn bản khác đã bị thay thế/bãi bỏ**.

### Nguyên lý
- Trong luật VN, văn bản mới thường có điều khoản hiệu lực dạng:
  *"Nghị định này thay thế Nghị định số 110/2004/NĐ-CP"*
- Ta tìm pattern này và **trích xuất số hiệu văn bản bị thay thế** → đưa vào danh sách loại bỏ

### Các pattern tìm kiếm
| Pattern | Ví dụ điển hình |
|---|---|
| thay thế ... số X | "thay thế Nghị định số 110/2004/NĐ-CP" |
| bãi bỏ ... số X | "bãi bỏ Thông tư số 04/2013/TT-BNV" |
| hết hiệu lực ... số X | "hết hiệu lực thi hành ... số 35/2005" |
| không còn hiệu lực ... số X | "không còn hiệu lực đối với..." |

In [4]:
# ─── Pattern nhận dạng điều khoản hiệu lực ───────────────────────────────────
# Mỗi pattern phải có group(1) = phần văn bản sau cụm từ (để trích số hiệu)
# Mở rộng: bắt thêm viết tắt NĐ/TT/QĐ/PL, dạng liệt kê nhiều văn bản
EFFECTIVITY_PATTERNS = [
    # "thay thế [các] [Nghị định/NĐ/Thông tư/TT/...] [số] X"
    r"thay thế\s+(?:các\s+)?(?:nghị định|thông tư|quyết định|pháp lệnh|luật|nghị quyết|bộ luật|n[đd]|tt|q[đd]|pl|qh)?\s*số\s+([\w/\-\.]+)",
    # "bãi bỏ [toàn bộ] [các] ... số X"
    r"bãi bỏ\s+(?:toàn bộ\s+)?(?:các\s+)?(?:nghị định|thông tư|quyết định|pháp lệnh|luật|nghị quyết|bộ luật|n[đd]|tt|q[đd]|pl|qh)?\s*số\s+([\w/\-\.]+)",
    # "hết hiệu lực ... số X"
    r"hết hiệu lực(?:[^.]{0,80}?)số\s+([\w/\-\.]+)",
    # "không còn hiệu lực ... số X"
    r"không còn hiệu lực(?:[^.]{0,80}?)số\s+([\w/\-\.]+)",
    # "thay thế cho ... số X"
    r"thay thế cho(?:[^.]{0,80}?)số\s+([\w/\-\.]+)",
    # "huỷ bỏ / hủy bỏ ... số X"
    r"h[uủ]y bỏ(?:[^.]{0,80}?)số\s+([\w/\-\.]+)",
    # "làm hết hiệu lực ... số X"  (variant ít gặp)
    r"làm hết hiệu lực(?:[^.]{0,80}?)số\s+([\w/\-\.]+)",
    # "đình chỉ thi hành ... số X"
    r"đình chỉ thi hành(?:[^.]{0,80}?)số\s+([\w/\-\.]+)",
    # "bãi bỏ ... quy định tại ... số X"
    r"bãi bỏ\s+(?:các\s+)?quy định(?:[^.]{0,60}?)số\s+([\w/\-\.]+)",
    # dạng liệt kê: "... số X, số Y, số Z" (lấy từng số hiệu)
    r"(?:thay thế|bãi bỏ|h[uủ]y bỏ)(?:[^.]{0,60}?)số\s+([\w/\-\.]+)(?:\s*,\s*số\s+([\w/\-\.]+))*",
]

# Pattern nhận dạng "số hiệu văn bản VN" đúng định dạng
# VD: 30/2020/NĐ-CP, 45/2019/QH14, 80/2015/QH13, 01/2011/QH13
DOC_NO_PATTERN = re.compile(
    r"^\d{1,3}/\d{4}/(?:[A-ZĐẮĂÂÁÀẢÃẠÊẾỀỂỄỆÔỐỒỔỖỘƠỚỜỞỠỢƯỨỪỬỮỰÍÌỈĨỊÚÙỦŨỤ\-]+)$",
    re.UNICODE
)

def normalize_vietnamese_doc_no(text: str) -> str:
    """Chuẩn hoá các biến thể Unicode của số hiệu văn bản."""
    if not text: return ""
    # Chuyển đổi các loại dấu gạch ngang (En Dash, Em Dash) về dấu gạch ngang chuẩn
    text = text.replace('–', '-').replace('—', '-')
    # Chuẩn hoá NĐ, QĐ thường gặp lỗi font
    text = text.upper().replace("NÐ", "NĐ").replace("QÐ", "QĐ")
    return text.strip()

def extract_replaced_doc_nos(content: str) -> list[str]:
    if not isinstance(content, str):
        return []

    found = []
    for pat in EFFECTIVITY_PATTERNS:
        for m in re.finditer(pat, content, re.IGNORECASE):
            # Lấy tất cả groups (hỗ trợ pattern liệt kê nhiều số hiệu)
            groups = [g for g in m.groups() if g]
            for raw in groups:
                raw = raw.strip().rstrip(".,;:)")
                normalized = normalize_vietnamese_doc_no(raw)
                # Chấp nhận dạng đầy đủ XX/YYYY/TT-CQ hoặc dạng ngắn XX/YYYY
                if re.match(r"\d{1,3}/\d{4}", normalized):
                    found.append(normalized)
    return list(set(found))


print("Test pattern:")
test_cases = [
    "Nghị định này thay thế Nghị định số 110/2004/NĐ-CP ngày 08 tháng 4 năm 2004",
    "bãi bỏ Thông tư số 04/2013/TT-BNV của Bộ trưởng",
    "Quyết định này hết hiệu lực kể từ ngày Thông tư số 01/2019/TT-BNV có hiệu lực",
    "hủy bỏ toàn bộ các quy định tại Thông tư số 07/2012/TT-BNV",
    "bãi bỏ NĐ số 24/2010/NĐ-CP, số 29/2012/NĐ-CP và số 56/2015/NĐ-CP",
    "đình chỉ thi hành Quyết định số 65/2017/NĐ-CP",
    "bãi bỏ các quy định về tuyển dụng tại Thông tư số 13/2010/TT-BNV",
]
for t in test_cases:
    result = extract_replaced_doc_nos(t)
    status = "✅" if result else "❌"
    print(f"  {status} {t[:70]}... → {result}")


Test pattern:
  ✅ Nghị định này thay thế Nghị định số 110/2004/NĐ-CP ngày 08 tháng 4 năm... → ['110/2004/NĐ-CP']
  ✅ bãi bỏ Thông tư số 04/2013/TT-BNV của Bộ trưởng... → ['04/2013/TT-BNV']
  ✅ Quyết định này hết hiệu lực kể từ ngày Thông tư số 01/2019/TT-BNV có h... → ['01/2019/TT-BNV']
  ✅ hủy bỏ toàn bộ các quy định tại Thông tư số 07/2012/TT-BNV... → ['07/2012/TT-BNV']
  ✅ bãi bỏ NĐ số 24/2010/NĐ-CP, số 29/2012/NĐ-CP và số 56/2015/NĐ-CP... → ['29/2012/NĐ-CP', '24/2010/NĐ-CP']
  ✅ đình chỉ thi hành Quyết định số 65/2017/NĐ-CP... → ['65/2017/NĐ-CP']
  ✅ bãi bỏ các quy định về tuyển dụng tại Thông tư số 13/2010/TT-BNV... → ['13/2010/TT-BNV']


## 3. Chạy Auto-detect trên toàn bộ Dataset

In [5]:
print("Đang quét content tìm văn bản bị thay thế...")

# Tập hợp số hiệu → tập hàng nguồn (doc_id của văn bản MỚI phát hiện ra)
auto_detected: dict[str, list[str]] = defaultdict(list)

EFFECTIVITY_KEYWORDS = [
    "hiệu lực", "bãi bỏ", "thay thế", "hủy bỏ", "huỷ bỏ",
    "không còn", "đình chỉ", "làm hết",
]

n_scanned = 0
for _, row in df.iterrows():
    content = str(row.get("content", "") or "")
    content_lower = content.lower()

    if not any(kw in content_lower for kw in EFFECTIVITY_KEYWORDS):
        continue

    n_scanned += 1
    replaced_nos = extract_replaced_doc_nos(content)

    source_id = str(row.get("id", "") or "")
    doc_id    = str(row.get("doc_id", "") or "")
    for r_no in replaced_nos:
        # Tránh văn bản tự tham chiếu chính mình
        if r_no and r_no not in source_id and r_no not in doc_id:
            auto_detected[r_no].append(source_id or doc_id)

print(f"Số điều khoản có từ khóa hiệu lực: {n_scanned:,}")
print(f"Số văn bản bị thay thế auto-detect: {len(auto_detected):,}")
print()
print("Top 20 văn bản bị mention nhiều nhất:")
top20 = sorted(auto_detected.items(), key=lambda x: len(x[1]), reverse=True)[:20]
for doc_no, sources in top20:
    n = len(sources)
    print(f"  {doc_no:<35} ← nhắc đến {n:>3}x  (bởi: {sources[0][:30]}{'...' if n > 1 else ''}{f' +{n-1} khác' if n > 1 else ''})")


Đang quét content tìm văn bản bị thay thế...
Số điều khoản có từ khóa hiệu lực: 62,480
Số văn bản bị thay thế auto-detect: 5,403

Top 20 văn bản bị mention nhiều nhất:
  13/2019/TT-BVHTTDL                  ← nhắc đến  29x  (bởi: 210/QĐ-UBND... +28 khác)
  05/2021/TT-BCT                      ← nhắc đến  14x  (bởi: 315/QĐ-UBND... +13 khác)
  51/2010/NĐ-CP                       ← nhắc đến  13x  (bởi: 78/2021/TT-BTC... +12 khác)
  15/2021/QĐ-UBND                     ← nhắc đến  12x  (bởi: 08/2023/QĐ-UBND... +11 khác)
  30/2001/QĐ-BTC                      ← nhắc đến  11x  (bởi: 78/2021/TT-BTC... +10 khác)
  13/2019/TT-                         ← nhắc đến   9x  (bởi: 1606/QĐ-UBND... +8 khác)
  34/2016/NĐ-CP                       ← nhắc đến   8x  (bởi: 1483/QĐ-BYT... +7 khác)
  155/2020/NĐ-CP                      ← nhắc đến   7x  (bởi: 65/2022/NĐ-CP... +6 khác)
  25/2022/TT-BNNPTNT                  ← nhắc đến   7x  (bởi: 525/QĐ-UBND... +6 khác)
  123/2020/NĐ-CP                      ← nhắc đến 

## 4. Tổng hợp Danh sách Loại bỏ

Merge blacklist cứng + auto-detect, áp ngưỡng confidence cho auto-detect.

In [6]:
# ── Ngưỡng confidence cho auto-detect ────────────────────────────────────────
# → giảm đáng kể false positive do 1-2 văn bản tham chiếu "nhắc đến" chứ không "thay thế"
AUTO_DETECT_MIN_SOURCES = 2

# ── Whitelist bảo vệ: không bao giờ loại các văn bản này dù auto-detect phát hiện ──
# Thường gặp trong các điều khoản "hết hiệu lực kể từ khi Luật X có hiệu lực"
# → câu này ở VB mới, nhưng X vẫn là luật hiện hành
NEVER_REMOVE_PATTERNS = [
    r"30/2020/NĐ-CP",    # Công tác văn thư — hiện hành
    r"80/2015/QH13",     # Luật BHVBQPPL hiện hành
    r"15/2020/QH14",     # Sửa đổi BHVBQPPL hiện hành
    r"45/2019/QH14",     # Bộ luật LĐ hiện hành
    r"22/2008/QH12",     # Luật CBCC hiện hành
    r"58/2010/QH12",     # Luật viên chức hiện hành
    r"01/2011/QH13",     # Luật lưu trữ hiện hành
    r"76/2015/QH13",     # Luật tổ chức CP hiện hành
    r"77/2015/QH13",     # Luật TCCQĐP hiện hành
    r"36/2018/QH14",     # Luật PCTN hiện hành
    r"02/2011/QH13",     # Luật khiếu nại hiện hành
    r"02/2011/QH11",     # Luật tố cáo hiện hành (khác QH13)
    r"34/2016/NĐ-CP",    # NĐ kỹ thuật soạn thảo VBQPPL
    r"138/2020/NĐ-CP",   # NĐ tuyển dụng CBCC hiện hành
    r"115/2020/NĐ-CP",   # NĐ tuyển dụng viên chức hiện hành
    r"61/2018/NĐ-CP",    # NĐ cơ chế một cửa hiện hành
    r"34/2019/NĐ-CP",    # NĐ CBCC cấp xã hiện hành
]

# ── Build danh sách loại bỏ cuối cùng ────────────────────────────────────────
REMOVE_LIST: dict[str, dict] = {}

# Lớp 1: Hardcode blacklist
for pattern, reason, replacement in BLACKLIST:
    REMOVE_LIST[pattern] = {
        "reason"      : reason,
        "replacement" : replacement,
        "source"      : "blacklist",
        "confidence"  : "HIGH",
    }

# Lớp 2: Auto-detect (chỉ lấy những cái đủ confidence)
auto_added = 0
for doc_no, sources in auto_detected.items():
    if len(sources) < AUTO_DETECT_MIN_SOURCES:
        continue

    # Không loại nếu nằm trong whitelist bảo vệ
    is_protected = any(
        re.search(safe_pat, doc_no, re.IGNORECASE)
        for safe_pat in NEVER_REMOVE_PATTERNS
    )
    if is_protected:
        continue

    # Không ghi đè blacklist
    already_in_blacklist = any(
        re.search(pat, doc_no)
        for pat in [item[0] for item in BLACKLIST]
    )
    if not already_in_blacklist:
        confidence = "HIGH" if len(sources) >= 5 else "MEDIUM"
        REMOVE_LIST[re.escape(doc_no)] = {
            "reason"      : f"Auto-detect: bị {len(sources)} VB khác thay thế/bãi bỏ",
            "replacement" : ", ".join(sorted(set(sources))[:3]),
            "source"      : "auto_detect",
            "confidence"  : confidence,
        }
        auto_added += 1

print(f"Danh sách loại bỏ tổng hợp:")
print(f"  Từ blacklist    : {len(BLACKLIST)}")
print(f"  Từ auto-detect  : {auto_added} (ngưỡng ≥{AUTO_DETECT_MIN_SOURCES} nguồn, có whitelist bảo vệ)")
print(f"  Tổng            : {len(REMOVE_LIST)}")


Danh sách loại bỏ tổng hợp:
  Từ blacklist    : 44
  Từ auto-detect  : 654 (ngưỡng ≥2 nguồn, có whitelist bảo vệ)
  Tổng            : 698


## 5. Áp dụng Filter — Loại bỏ Văn bản Hết hiệu lực

In [7]:
n_before = len(df)

# Build mask: True = hàng CẦN BỎ
# Ghép tất cả pattern thành 1 regex lớn để match nhanh
combined_pattern = "|".join(f"(?:{p})" for p in REMOVE_LIST.keys())

id_col = "id" if "id" in df.columns else "source_doc_no"
mask_obsolete = df[id_col].str.contains(
    combined_pattern, na=False, regex=True, flags=re.IGNORECASE
)

# Thống kê theo từng văn bản
print("Chi tiết từng văn bản bị loại:")
print(f"{'Số hiệu':<35} {'Số điều':>8} {'Nguồn':<15} {'Conf.':<8} Lý do")
print("-" * 100)

total_removed_by_doc: dict[str, int] = {}
for pattern, meta in REMOVE_LIST.items():
    doc_mask = df[id_col].str.contains(pattern, na=False, regex=True, flags=re.IGNORECASE)
    n_rows = doc_mask.sum()
    if n_rows > 0:
        # Lấy số hiệu thực tế (không phải pattern)
        actual_ids = df.loc[doc_mask, id_col].dropna().unique()
        for actual_id in actual_ids[:2]:  # hiển thị tối đa 2
            total_removed_by_doc[actual_id] = n_rows
            print(f"  {actual_id:<33} {n_rows:>8,}  {meta['source']:<15} {meta['confidence']:<8} {meta['reason'][:50]}")

df_active = df[~mask_obsolete].reset_index(drop=True)
n_after = len(df_active)

print()
print(f"Trước filter : {n_before:,} hàng")
print(f"Sau filter   : {n_after:,} hàng")
print(f"Đã loại bỏ  : {n_before - n_after:,} hàng ({(n_before - n_after)/n_before*100:.1f}%)")

Chi tiết từng văn bản bị loại:
Số hiệu                              Số điều Nguồn           Conf.    Lý do
----------------------------------------------------------------------------------------------------
  110/2004/NĐ-CP                          34  blacklist       HIGH     Thay bởi NĐ 30/2020
  109/2010/NĐ-CP                          36  blacklist       HIGH     Thay bởi NĐ 30/2020
  09/2010/NĐ-CP                           36  blacklist       HIGH     Thay bởi NĐ 30/2020
  04/2013/TT-BNV                          38  blacklist       HIGH     Thay bởi TT 01/2019/TT-BNV
  07/2012/TT-BNV                          27  blacklist       HIGH     Thay bởi TT 01/2019/TT-BNV
  01/2011/TT-BNV                          19  blacklist       HIGH     Thay bởi NĐ 30/2020 + TT mới
  155/2005/NĐ-CP                          14  blacklist       HIGH     Thay bởi NĐ 110/2004 → 30/2020
  02/2012/TT-BNV                           8  blacklist       HIGH     TT lưu trữ cũ — thay bởi TT 02/2021/TT-BNV
  09/20

## 6. Export Review File — Để Human Verify

In [8]:
# Tạo bảng review: các văn bản bị loại + metadata
review_rows = []

# Văn bản bị loại bởi blacklist/auto-detect
for pattern, meta in REMOVE_LIST.items():
    doc_mask = df[id_col].str.contains(pattern, na=False, regex=True, flags=re.IGNORECASE)
    if doc_mask.sum() == 0:
        # Pattern không match gì → có thể sai hoặc văn bản không có trong dataset
        review_rows.append({
            "pattern"      : pattern,
            "actual_id"    : "(không có trong dataset)",
            "n_rows_removed": 0,
            "source"       : meta["source"],
            "confidence"   : meta["confidence"],
            "reason"       : meta["reason"],
            "replacement"  : meta["replacement"],
            "action"       : "SKIP — không có trong data",
        })
    else:
        actual_ids = df.loc[doc_mask, id_col].dropna().unique()
        for actual_id in actual_ids:
            n = df.loc[df[id_col] == actual_id].shape[0]
            review_rows.append({
                "pattern"       : pattern,
                "actual_id"     : actual_id,
                "n_rows_removed": n,
                "source"        : meta["source"],
                "confidence"    : meta["confidence"],
                "reason"        : meta["reason"],
                "replacement"   : meta["replacement"],
                "action"        : "REMOVED",
            })

# Auto-detect với confidence thấp (< ngưỡng) — để tham khảo, KHÔNG loại tự động
for doc_no, sources in auto_detected.items():
    if len(sources) < AUTO_DETECT_MIN_SOURCES:
        review_rows.append({
            "pattern"       : re.escape(doc_no),
            "actual_id"     : doc_no,
            "n_rows_removed": 0,
            "source"        : "auto_detect_LOW",
            "confidence"    : "LOW",
            "reason"        : f"Chỉ bị 1 văn bản nhắc đến ({sources[0]})",
            "replacement"   : sources[0] if sources else "",
            "action"        : "KEPT — confidence thấp, cần review thủ công",
        })

df_review = pd.DataFrame(review_rows).sort_values(
    ["source", "confidence", "n_rows_removed"],
    ascending=[True, True, False]
)

DATASET_DIR.mkdir(parents=True, exist_ok=True)
df_review.to_csv(REVIEW_PATH, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu review file: {REVIEW_PATH}")
print(f"   {len(df_review)} entries để review")
print()
print("Tóm tắt theo action:")
print(df_review["action"].value_counts().to_string())

✅ Đã lưu review file: dataset/processed/obsolete_review.csv
   5655 entries để review

Tóm tắt theo action:
action
KEPT — confidence thấp, cần review thủ công    4742
REMOVED                                         757
SKIP — không có trong data                      156


## 7. Kiểm tra Coverage — Văn bản Active còn lại

In [9]:
# Đảm bảo các văn bản quan trọng hiện hành vẫn còn đủ
ACTIVE_LAWS_CHECK = {
    "NĐ 30/2020 (Văn thư)"            : r"30/2020/NĐ-CP",
    "Luật Ban hành VBQPPL 80/2015"     : r"80/2015/QH13",
    "Luật sửa đổi BHVBQPPL 15/2020"   : r"15/2020/QH14",
    "Bộ luật Lao động 45/2019"         : r"45/2019/QH14",
    "Luật Cán bộ, Công chức 22/2008"  : r"22/2008/QH12",
    "Luật Viên chức 58/2010"           : r"58/2010/QH12",
    "Luật Lưu trữ 01/2011"            : r"01/2011/QH13",
    "Luật Tổ chức Chính phủ 76/2015"  : r"76/2015/QH13",
    "Luật TCCQĐP 77/2015"             : r"77/2015/QH13",
    "Luật Phòng chống tham nhũng 2018": r"36/2018/QH14",
    "Luật Khiếu nại 02/2011"          : r"02/2011/QH13",
    "NĐ 138/2016 (Tổ chức Chính phủ)" : r"138/2016/NĐ-CP",
    "NĐ 34/2019 (CBCC cấp xã)"        : r"34/2019/NĐ-CP",
}

print(f"{'Văn bản hiện hành':<45} {'Điều khoản':>12} {'Trạng thái'}")
print("-" * 72)

all_ok = True
for label, pattern in ACTIVE_LAWS_CHECK.items():
    hits = df_active[id_col].str.contains(pattern, case=False, na=False).sum()
    if hits == 0 and "name" in df_active.columns:
        hits = df_active["name"].str.contains(pattern, case=False, na=False).sum()
    status = "✅" if hits > 0 else "❌ MISSING — kiểm tra lại!"
    if hits == 0:
        all_ok = False
    print(f"  {label:<43} {hits:>12,}   {status}")

print()
if all_ok:
    print("✅ Tất cả văn bản hiện hành đều còn đủ sau filter.")
else:
    print("⚠️  Có văn bản hiện hành bị loại nhầm — kiểm tra lại BLACKLIST!")

Văn bản hiện hành                               Điều khoản Trạng thái
------------------------------------------------------------------------
  NĐ 30/2020 (Văn thư)                                  63   ✅
  Luật Ban hành VBQPPL 80/2015                         174   ✅
  Luật sửa đổi BHVBQPPL 15/2020                          3   ✅
  Bộ luật Lao động 45/2019                             220   ✅
  Luật Cán bộ, Công chức 22/2008                        89   ✅
  Luật Viên chức 58/2010                                61   ✅
  Luật Lưu trữ 01/2011                                  42   ✅
  Luật Tổ chức Chính phủ 76/2015                        48   ✅
  Luật TCCQĐP 77/2015                                  143   ✅
  Luật Phòng chống tham nhũng 2018                      96   ✅
  Luật Khiếu nại 02/2011                                70   ✅
  NĐ 138/2016 (Tổ chức Chính phủ)                       50   ✅
  NĐ 34/2019 (CBCC cấp xã)                               4   ✅

✅ Tất cả văn bản hiện hành đều còn đủ

## 8. Lưu Output & Báo cáo tổng hợp

In [10]:
# Lưu parquet
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_active.to_parquet(OUTPUT_PATH, engine="pyarrow", index=False)

size_mb = OUTPUT_PATH.stat().st_size / 1024 / 1024

# Báo cáo
report_lines = [
    "=" * 70,
    "   BÁO CÁO OBSOLETE FILTER",
    "=" * 70,
    f"Input : {INPUT_PATH}  ({n_before:,} hàng)",
    f"Output: {OUTPUT_PATH}  ({n_after:,} hàng)",
    f"Đã loại bỏ: {n_before - n_after:,} hàng ({(n_before-n_after)/n_before*100:.1f}%)",
    "",
    "Nguồn loại bỏ:",
    f"  Blacklist cứng   : {len(BLACKLIST)} patterns",
    f"  Auto-detect HIGH  : {sum(1 for m in REMOVE_LIST.values() if m['source']=='auto_detect' and m['confidence']=='HIGH')} patterns",
    f"  Auto-detect MEDIUM: {sum(1 for m in REMOVE_LIST.values() if m['source']=='auto_detect' and m['confidence']=='MEDIUM')} patterns",
    "",
    "Phân bố type_normalized sau filter:",
]
for t, c in df_active["type_normalized"].value_counts().items():
    report_lines.append(f"  {t:<20} {c:>8,}  ({c/n_after*100:.1f}%)")

report_lines += [
    "",
    f"Review file: {REVIEW_PATH}",
    "=" * 70,
    "Bước tiếp theo: Chạy filterData.ipynb với INPUT_PATH trỏ vào:",
    f"  INPUT_PATH = Path('{OUTPUT_PATH}')",
]

report_text = "\n".join(report_lines)
print(report_text)

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(report_text)
print(f"\n✅ Báo cáo lưu tại: {REPORT_PATH}")

   BÁO CÁO OBSOLETE FILTER
Input : dataset/processed/legal_dataset_processed.parquet  (392,079 hàng)
Output: dataset/processed/legal_dataset_active.parquet  (368,865 hàng)
Đã loại bỏ: 23,214 hàng (5.9%)

Nguồn loại bỏ:
  Blacklist cứng   : 44 patterns
  Auto-detect HIGH  : 31 patterns
  Auto-detect MEDIUM: 623 patterns

Phân bố type_normalized sau filter:
  THÔNG TƯ              130,223  (35.3%)
  QUYẾT ĐỊNH             81,737  (22.2%)
  NGHỊ ĐỊNH              64,637  (17.5%)
  NGHỊ QUYẾT             50,914  (13.8%)
  LUẬT                   30,208  (8.2%)
  PHÁP LỆNH               6,798  (1.8%)
  KHÁC                    4,265  (1.2%)
  VĂN BẢN KHÁC               56  (0.0%)
  KHÔNG RÕ                   27  (0.0%)

Review file: dataset/processed/obsolete_review.csv
Bước tiếp theo: Chạy filterData.ipynb với INPUT_PATH trỏ vào:
  INPUT_PATH = Path('dataset/processed/legal_dataset_active.parquet')

✅ Báo cáo lưu tại: dataset/processed/obsolete_report.txt


---
## 9. Thay đổi cần làm ở notebook tiếp theo

Sau khi chạy notebook này, cần sửa **1 dòng** trong `filterData.ipynb`:

```python
# TRƯỚC
INPUT_PATH  = DATASET_DIR / "legal_dataset_processed.parquet"

# SAU — trỏ vào output của notebook này
INPUT_PATH  = DATASET_DIR / "legal_dataset_active.parquet"
```

## 10. Hướng dẫn maintain BLACKLIST

Khi phát hiện thêm văn bản hết hiệu lực, thêm vào `BLACKLIST` ở Cell 1:

```python
# Thêm 1 entry:
(r"<số_hiệu_regex>",  "Thay bởi <văn_bản_mới>",  "<số_hiệu_mới>"),

# Ví dụ:
(r"29/2012/NĐ-CP",   "Thay bởi NĐ 138/2020/NĐ-CP",  "138/2020/NĐ-CP"),
```

